# Import libraries

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from googleapiclient.discovery import build
from transformers import pipeline
from dotenv import load_dotenv
from datetime import datetime, timezone
from dateutil.relativedelta import relativedelta

# Setup

In [ ]:
load_dotenv()

API_KEY = os.getenv('YOUTUBE_API_KEY') 
MAX_VIDEOS_PER_MONTH = 10
MAX_COMMENTS_PER_VIDEO = 100    
SEARCH_QUERY = ''
START_DATE = datetime(2025, 10, 1, tzinfo=timezone.utc)
END_DATE = datetime(2025, 12, 31, tzinfo=timezone.utc)

SEED_TERMS = [
    "AI",
    "artificial intelligence",
]


if not API_KEY:
    print("WARNING: API Key not found. Please check your .env file.")
else:
    print("Configuration loaded. Ready to proceed.")

In [ ]:
youtube = build('youtube', 'v3', developerKey=API_KEY)
print("Connected to YouTube API successfully.")

print("Loading emotion analysis model (this might take a moment)...")
# emotion_classifier = pipeline("text-classification", model="j-hartmann/emotion-english-distilroberta-base")
# print("Model loaded successfully!")

# Retrieve comments

In [ ]:
def get_video_ids(query, max_results, published_after, published_before):
    """Searches YouTube for the most viewed videos within a specific timeframe."""
    request = youtube.search().list(
        part="id",
        q=query,
        type="video",
        order="viewCount", 
        publishedAfter=published_after.isoformat(),
        publishedBefore=published_before.isoformat(),
        maxResults=max_results
    )
    response = request.execute()
    return [item['id']['videoId'] for item in response.get('items', [])]

def get_video_comments(video_id, max_results):
    """Fetches top comments from a specific video."""
    comments = []
    try:
        request = youtube.commentThreads().list(
            part="snippet",
            videoId=video_id,
            maxResults=max_results,
            textFormat="plainText"
        )
        response = request.execute()
        
        for item in response.get('items', []):
            comment = item['snippet']['topLevelComment']['snippet']['textDisplay']
            comments.append(comment)
    except Exception as e:
        pass
        
    return comments

In [ ]:
all_data = []
unique_video_ids = set()  # To track unique video IDs and avoid duplicates

for term in SEED_TERMS:
    SEARCH_QUERY = term
    current_date = START_DATE   
    print(f"\n=== Analyzing term: '{term}' ===")
    print(f"Starting temporal analysis from {START_DATE.year} to {END_DATE.year}...")

    while current_date < END_DATE:
        next_month = current_date + relativedelta(months=1)
        
        month_label = current_date.strftime('%Y-%m')
        print(f"Processing {month_label}...")
        
        # 1. Get top 10 most viewed videos for THIS month
        video_ids = get_video_ids(
            query=SEARCH_QUERY, 
            max_results=MAX_VIDEOS_PER_MONTH, 
            published_after=current_date, 
            published_before=next_month
        )
                
        # 2. Analyze comments for these videos
        if video_ids:
            for video_id in video_ids:
                
                comments = get_video_comments(video_id, MAX_COMMENTS_PER_VIDEO)
                
                for comment in comments:
                    # Truncate to avoid model token limits
                    safe_comment = comment[:500] 
                    # result = emotion_classifier(safe_comment)[0]

                    if video_id not in unique_video_ids:
                        unique_video_ids.add(video_id)    
                        all_data.append({
                            "Month": month_label,
                            "Video_ID": video_id,
                            "Comment": comment
                        })

        else:
            print(f"  -> No videos found for {month_label}.")
                
        # Move to the next month
        current_date = next_month

# Eliminate duplicate video entries (if any)


# Create the final DataFrame

df = pd.DataFrame(all_data)
print("\nDone! Analyzed a total of", len(df), "comments across the timeline.")

# Convert 'Month' to an actual datetime object for easier plotting
df['Month'] = pd.to_datetime(df['Month'])

# Display the first few rows
df.head()

In [ ]:
df.head(50)